<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.1 — Regression as an Optimization Problem

A regression model is an optimization problem at heart: choose the parameters (like $a$ and $b$) that make the
distance between the actual and predicted values as small as possible.

When you write that optimization model yourself, **you** make two choices that software usually makes for you:

1. **The form of the model**: a straight line, a power curve, a polynomial...
2. **The error you minimize**: squared error, absolute error...

Change either one and you get a **different answer**, and some curves will simply look better to the eye than
others. Tonight you will build the model once, then play: every scenario you run is added to one picture so you
can compare them side by side.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · The pharmacy problem

A pharmacy has hired you to predict weekly **revenue** from the number of **hours** the store is open. Here are
ten stores (168 hours is a store open 24/7):

| Hours | 40 | 44 | 48 | 48 | 60 | 70 | 72 | 90 | 100 | 168 |
|---|---|---|---|---|---|---|---|---|---|---|
| Revenue (\$) | 5,958 | 6,662 | 6,004 | 6,011 | 7,250 | 8,632 | 6,964 | 11,097 | 9,107 | 11,498 |

As in M4.1, we keep the data separate from the model: `X` holds the hours, `Z` holds the actual revenue.

In [ ]:
# read the data
pharmacy_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/7_Nonlinear/data/pharmacy.csv')

# X = hours (the input), Z = revenue (the actual value we want to predict)
X = []
Z = []
for i in range(len(pharmacy_df)):
    X.append(float(pharmacy_df.loc[i, 'hours']))
    Z.append(float(pharmacy_df.loc[i, 'revenue']))

# this is how many observations (stores) we have
n_points = len(X)

print("X (hours):  ", X)
print("Z (revenue):", Z)
print("n_points =", n_points)

In [ ]:
# look before you model
scatter(X, Z, color='black')
xlabel('Hours open per week')
ylabel('Weekly revenue ($)')
title('Revenue vs. hours: ten pharmacies')
show()

# Part 2 · The model on paper

### The decision variables

- The **parameters** of the model: $a$ and $b$ (and $c$ if the model needs a third one). These are what the solver chooses.
- One **placeholder prediction per store**: $y_0, y_1, \dots, y_9$. This is the big leap. On its own, each $y_i$
  could take on **any** value. Ten stores, ten placeholders.

### The constraints: where the form of the model lives

Here is the sneaky part: the regression formula is **not** in the objective. It is in the **constraints**, one per
store. Each constraint forces that store's placeholder to follow the model:

| Model form | One constraint per store $i$ | Parameters |
|---|---|---|
| Linear | $y_i = a + b\,x_i$ | $a, b$ |
| Power | $y_i = a \cdot x_i^{\,b}$ | $a, b$ |
| Quadratic (a polynomial) | $y_i = a + b\,x_i + c\,x_i^2$ | $a, b, c$ |

Notice that $a$ and $b$ are the **same** in every constraint: all ten stores share one model.

### The objective: the error you minimize

Now the objective only has to compare each placeholder $y_i$ with the actual value $z_i$:

| Error | Objective |
|---|---|
| Squared error (the usual one) | $\min \sum_{i} (y_i - z_i)^2$ |
| Absolute error | $\min \sum_{i} \lvert y_i - z_i \rvert$ |

Squared error punishes one big miss much more than several small ones. Absolute error treats every dollar of miss
the same.

### Putting it together (linear model, squared error)

$$\min \sum_{i=0}^{9} (y_i - z_i)^2 \quad \text{subject to} \quad y_i = a + b\,x_i \;\; \text{for every store } i, \qquad a, b, y_i \in \mathbb{R}$$

**The answer to check against.** `scipy` fits the ordinary least-squares line in one line of code. Our
optimization model (linear form, squared error) must give exactly the same $a$ and $b$. If it does, the model is right.

In [ ]:
from scipy import stats

scipy_fit = stats.linregress(X, Z)
print("scipy:  a = %.2f   b = %.4f" % (scipy_fit.intercept, scipy_fit.slope))

**The scoreboard.** An empty dictionary. Every scenario you run adds one entry with what the model found
(`scoreboard[scenario] = ...`), and Part 5 draws all of them on one picture. Run this cell **once**: running it
again wipes the scoreboard clean.

In [ ]:
# scenario name -> what the model found
scoreboard = {}

# Part 3 · The scenario

Leave it alone for the first run (the base scenario: linear model, squared error). Then try the others **one at a
time**: remove the `#` from that scenario's lines, click on the cell, and choose **Runtime → Run cell and below**
(Ctrl+F10). Put the `#` back when you are done.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
scenario = "1 linear, squared error"
model_form = "linear"        # "linear", "power" or "quadratic"
error_type = "squared"       # "squared" or "absolute"
never_overpredict = False    # a business rule (Scenario 6)

# Scenario 2: same line, absolute error
# scenario = "2 linear, absolute error"
# error_type = "absolute"

# Scenario 3: a power model
# scenario = "3 power, squared error"
# model_form = "power"

# Scenario 4: a power model with absolute error
# scenario = "4 power, absolute error"
# model_form = "power"
# error_type = "absolute"

# Scenario 5: a polynomial (quadratic) model
# scenario = "5 quadratic, squared error"
# model_form = "quadratic"

# Scenario 6: finance wants a budget-safe line that never predicts more than a store actually made
# scenario = "6 linear, never over-predict"
# never_overpredict = True

# Scenario 7: your own idea (mix and match!)


# Part 4 · Build it, one piece at a time

**4a · The decision variables.** The parameters $a$ and $b$ (plus $c$ for the quadratic), and the ten placeholder
predictions `model.y[i]`. Nonlinear solvers need a sensible starting point (the hill climber in the fog!), so we
initialize everything.

Look at the printout: each `y` has **no bounds and no rule** yet. Right now it could be anything.

In [ ]:
# ---------- 4a. the decision variables ----------
model = ConcreteModel()

# the parameters of the model
model.a = Var(domain=Reals, initialize=1000)
model.b = Var(domain=Reals, initialize=1)
if model_form == "power":
    model.b = 0.5        # a power model starts near a square root
if model_form == "quadratic":
    model.c = Var(domain=Reals, initialize=0)

# one placeholder prediction per store - on its own, each one can be anything
model.y = Var(range(n_points), domain=Reals)
for i in range(n_points):
    model.y[i] = 0.0

model.y.pprint()

**4b · The constraints: knob 1, the form of the model.** One constraint per store. This is where each
placeholder `y[i]` gets **forced** to follow the model. Look at what Pyomo prints: the prediction minus the model
has to equal 0, and the same $a$ and $b$ appear in every row.

In [ ]:
# ---------- 4b. one constraint per store: the form of the model ----------
model.constraints = ConstraintList()

for i in range(n_points):
    x = X[i]
    if model_form == "linear":
        model.constraints.add(model.y[i] == model.a + model.b * x)
    if model_form == "power":
        model.constraints.add(model.y[i] == model.a * x ** model.b)
    if model_form == "quadratic":
        model.constraints.add(model.y[i] == model.a + model.b * x + model.c * x ** 2)

# show the first three constraints (there are n_points of them)
print(model_form, "model:", len(model.constraints), "constraints, one per store. The first three:")
for k in [1, 2, 3]:
    print("  ", model.constraints[k].expr)

**4c · The objective: knob 2, the error.** The `+=` loop adds one store's error at a time.

For **absolute error** we cannot use `abs()`: the solver follows the slope of the objective downhill, and a V has
a sharp point with no slope at the bottom, exactly where we want to end up. So we make one more variable per store,
`model.d[i]`, and give it **two floors**: `d >= miss` and `d >= -miss`.

| the store's miss ($y_i - z_i$) | floor 1: d >= miss | floor 2: d >= -miss | smallest d allowed |
|---|---|---|---|
| +300 (we over-predicted) | d >= 300 | d >= -300 | **300** |
| -200 (we under-predicted) | d >= -200 | d >= 200 | **200** |

One of the two floors is always the positive size of the miss. Because we **minimize** the total of the `d`'s, the
solver pushes every `d` down until it hits the higher floor, which is exactly $\lvert y_i - z_i \rvert$.

In [ ]:
# ---------- 4c. the objective: the error you minimize ----------
obj_expr = 0

if error_type == "squared":
    for i in range(n_points):
        obj_expr += (model.y[i] - Z[i]) ** 2

if error_type == "absolute":
    # d[i] = |y[i] - Z[i]| at the optimum (two floors per store)
    model.d = Var(range(n_points), domain=NonNegativeReals)
    for i in range(n_points):
        model.constraints.add(model.d[i] >= model.y[i] - Z[i])
        model.constraints.add(model.d[i] >= -(model.y[i] - Z[i]))
        obj_expr += model.d[i]

model.error = Objective(expr=obj_expr, sense=minimize)
print("minimize", error_type, "error:")
print(obj_expr)

**4d · A business rule (Scenario 6).** Plain regression has no way to say "never predict more than a store
actually made". In an optimization model it is one more constraint per store: $y_i \le z_i$. (Adding a constraint
can never make the error *smaller*; watch what it costs.)

In [ ]:
# ---------- 4d. a business rule ----------
if never_overpredict:
    for i in range(n_points):
        model.constraints.add(model.y[i] <= Z[i])
    print("rule: never over-predict (", n_points, "more constraints )")
else:
    print("no business rule in this scenario")

**4e · Solve it and read the results.** We report the fitted model with its numbers, the total squared
error (one yardstick for every scenario), and a prediction for a store open 120 hours, a store we have no data for.

In [ ]:
# ---------- 4e. solve and report ----------
results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
print("SCENARIO:", scenario)
print("Termination:", results.solver.termination_condition)

a = model.a()
b = model.b()
c = 0
if model_form == "quadratic":
    c = model.c()

if model_form == "linear":
    print("revenue = %.2f + %.4f * hours" % (a, b))
if model_form == "power":
    print("revenue = %.2f * hours ** %.4f" % (a, b))
if model_form == "quadratic":
    print("revenue = %.2f + %.4f * hours + %.6f * hours**2" % (a, b, c))

# the same yardstick for every scenario
sse = 0
for i in range(n_points):
    sse += (model.y[i]() - Z[i]) ** 2

if model_form == "linear":
    pred_120 = a + b * 120
if model_form == "power":
    pred_120 = a * 120 ** b
if model_form == "quadratic":
    pred_120 = a + b * 120 + c * 120 ** 2

print("total squared error = {:,.0f}".format(sse))
print("predicted revenue for a store open 120 hours = ${:,.0f}".format(pred_120))

# add this scenario to the scoreboard: scoreboard[key] = value
scoreboard[scenario] = {'form': model_form, 'error': error_type, 'a': a, 'b': b, 'c': c,
                        'squared error': sse, 'revenue at 120 hours': pred_120}

# Part 5 · Every scenario, side by side

The table and the picture grow every time you run a scenario.

In [ ]:
# the scoreboard as a table
pd.DataFrame(scoreboard).T

In [ ]:
# every scenario's curve on one picture
grid = np.linspace(30, 175, 200)

scatter(X, Z, color='black', zorder=3, label='the ten stores')
for name in scoreboard:
    fit = scoreboard[name]
    if fit['form'] == 'linear':
        curve = fit['a'] + fit['b'] * grid
    if fit['form'] == 'power':
        curve = fit['a'] * grid ** fit['b']
    if fit['form'] == 'quadratic':
        curve = fit['a'] + fit['b'] * grid + fit['c'] * grid ** 2
    plot(grid, curve, label=name)
xlabel('Hours open per week')
ylabel('Weekly revenue ($)')
title('Same data, different choices, different answers')
legend(fontsize=8)
show()

### Your turn

Run scenarios 1 to 6, then answer in a text cell:

1. Does Scenario 1 match `scipy` exactly? Why does that matter?
2. Linear model, squared vs. absolute error (Scenarios 1 and 2): how did $a$ and $b$ change, and why?
3. Which curve looks best to your eye? Which one has the lowest squared error? Are they the same one?
4. The power model's $b$ is below 1. What does that say about the value of staying open longer?
5. Scenario 6 makes the squared error worse. Why is that guaranteed, and why would finance still want it?

# Appendix · Each model, written cleanly

The notebook above uses switches (`model_form`, `error_type`) so one set of cells can run every scenario. In the
wild you would usually write just the one model you need. Here is each one on its own: the same pieces, nothing
else. Each cell runs by itself after Part 1 (it only needs `X`, `Z` and `n_points`).

## A1 · Linear model, squared error (ordinary least squares)

$$y_i = a + b\,x_i \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i])

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours" % (m.a(), m.b()))

## A2 · Linear model, absolute error

$$y_i = a + b\,x_i \qquad \min \sum_i d_i \;\text{ with }\; d_i \ge y_i - z_i,\; d_i \ge -(y_i - z_i)$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.y = Var(range(n_points))
m.d = Var(range(n_points), domain=NonNegativeReals)     # d[i] = |y[i] - Z[i]|

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i])
    m.fit.add(m.d[i] >= m.y[i] - Z[i])
    m.fit.add(m.d[i] >= -(m.y[i] - Z[i]))

error = 0
for i in range(n_points):
    error += m.d[i]
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours" % (m.a(), m.b()))

## A3 · Power model, squared error

$$y_i = a \cdot x_i^{\,b} \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=0.5)       # start near a square root
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a * X[i] ** m.b)

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f * hours ** %.4f" % (m.a(), m.b()))

## A4 · Power model, absolute error

$$y_i = a \cdot x_i^{\,b} \qquad \min \sum_i d_i \;\text{ with }\; d_i \ge y_i - z_i,\; d_i \ge -(y_i - z_i)$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=0.5)
m.y = Var(range(n_points))
m.d = Var(range(n_points), domain=NonNegativeReals)

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a * X[i] ** m.b)
    m.fit.add(m.d[i] >= m.y[i] - Z[i])
    m.fit.add(m.d[i] >= -(m.y[i] - Z[i]))

error = 0
for i in range(n_points):
    error += m.d[i]
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f * hours ** %.4f" % (m.a(), m.b()))

## A5 · Quadratic (polynomial) model, squared error

$$y_i = a + b\,x_i + c\,x_i^2 \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.c = Var(initialize=0)
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i] + m.c * X[i] ** 2)

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours + %.6f * hours**2" % (m.a(), m.b(), m.c()))

## A6 · Linear model, squared error, never over-predict

$$y_i = a + b\,x_i \;\text{ and }\; y_i \le z_i \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i])
    m.fit.add(m.y[i] <= Z[i])        # the business rule: never predict more than the store made

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours" % (m.a(), m.b()))

## Bottom line

A regression is an optimization model:

- the **decision variables** are the parameters ($a$, $b$, ...) plus one placeholder prediction per row,
- the **constraints** force each placeholder to follow the **form** of the model,
- the **objective** is the **error** you choose to minimize,
- and anything else the business knows can be added as **one more constraint**.

Change the form or the error and you get a different answer. Part of the job is choosing the one that makes sense.